# مشروع استخراج وتصحيح النصوص اليدوية — النسخة الشاملة Ultimate v4.0

> **Handwritten OCR System** — TrOCR + EasyOCR + ar-corrector

| Feature | Description |
|---|---|
| 🌐 Languages | Arabic, English, German |
| 🧠 Models | TrOCR (David-Magdy/TR_OCR_LARGE), EasyOCR, Tesseract |
| 📄 Input | PDF or Images (PNG, JPG, etc.) |
| 🔄 Ensemble | Smart OCR selection with confidence-based routing |
| ✏️ Correction | ar-corrector, spellchecker, user feedback loop |
| 📈 Self-Improvement | LoRA fine-tuning, active learning |
| 📊 Metrics | WER, CER tracking with plots |
| 💾 Export | CSV, XLSX, TXT, JSONL training data |
| 🚀 Deploy | Gradio UI (local + HuggingFace Spaces) |

**GitHub:** https://github.com/DrAbdulmalek/handwriting-ocr  
**HF Space:** https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr

---


In [ ]:
# 📦 تثبيت المكتبات المطلوبة — Install required libraries
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara tesseract-ocr-deu -qq

!pip install -q \
    pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm


In [ ]:
# 🔧 الاستيرادات والإعداد — Imports & Setup
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
import multiprocessing as mp
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import gradio as gr
import pytesseract
import easyocr

from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from tqdm import tqdm

# محاولة استيراد مصحح العربية — Try importing Arabic corrector
try:
    from ar_corrector.corrector import Corrector
    _AR_CORRECTOR_AVAILABLE = True
except ImportError:
    _AR_CORRECTOR_AVAILABLE = False

# التحقق من بيئة كولاب — Check Colab environment
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    try:
        _HF_TOKEN = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = _HF_TOKEN
        print('✅ HF_TOKEN loaded from Colab Secrets')
    except Exception:
        _HF_TOKEN = None
        print('ℹ️ No HF_TOKEN found in Colab Secrets')
    IN_COLAB = True
except ImportError:
    _HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ Running in local environment')

# تثبيت بذرة الكشف عن اللغة للنتائج المتسقة — Seed for consistent language detection
DetectorFactory.seed = 0
print('✅ All imports complete')


In [ ]:
# ⚙️ إعدادات المشروع — Project Configuration

@dataclass
class Config:
    """إعدادات شاملة للمشروع — Comprehensive project settings"""
    # --- المسارات الأساسية — Core paths ---
    project_root: str = '/content/handwriting-ocr'
    pdf_path: str = ''
    db_name: str = 'handwriting_ocr.db'

    # --- النماذج — Models ---
    trocr_model_name: str = 'David-Magdy/TR_OCR_LARGE'
    hf_token: str = ''
    hf_dataset_repo: str = ''

    # --- اللغات المدعومة — Supported languages ---
    ocr_languages: list = field(default_factory=lambda: ['en', 'ar', 'de'])
    tesseract_lang_config: str = 'ara+eng+deu'

    # --- إعدادات الأداء — Performance settings ---
    dpi: int = 300
    use_gpu: bool = True
    trocr_batch_size: int = 16
    num_beams: int = 4
    max_text_length: int = 64

    # --- عتبات الثقة — Confidence thresholds ---
    easy_conf_threshold: float = 0.80
    trocr_default_conf: float = 0.70
    low_conf_threshold: float = 0.50

    # --- معالجة الصور — Image preprocessing ---
    clahe_clip: float = 2.0
    clahe_tile: tuple = (8, 8)
    denoise_h: int = 20
    enable_deskew: bool = True

    # --- التدريب — Training ---
    finetune_min_samples: int = 50
    finetune_epochs: int = 5
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    learning_rate: float = 3e-4

    # --- التعلم النشط — Active Learning ---
    al_min_new_corrections: int = 50

    # --- التصدير — Export ---
    auto_export_after_run: bool = True

    # --- غراديو — Gradio ---
    gradio_share: bool = True

    # --- التصحيح — Corrections ---
    correction_min_votes: int = 2

    # --- صفحات المعالجة — Page range ---
    pages_start: int = 1
    pages_end: int = None  # None = all pages

    # --- السجلات — Logging ---
    log_level: str = 'INFO'

    # --- خصائص المسارات — Path properties ---
    @property
    def root(self) -> Path:
        return Path(self.project_root)

    @property
    def db_path(self) -> Path:
        return self.root / self.db_name

    @property
    def logs_dir(self) -> Path:
        return self.root / 'logs'

    @property
    def exports_dir(self) -> Path:
        return self.root / 'exports'

    @property
    def cache_dir(self) -> Path:
        return self.root / 'cache'

    @property
    def artifacts_dir(self) -> Path:
        return self.root / 'artifacts'

    @property
    def backups_dir(self) -> Path:
        return self.root / 'backups'

    @property
    def tensorboard_dir(self) -> Path:
        return self.root / 'tensorboard'

    @property
    def feedback_csv(self) -> Path:
        return self.root / 'feedback.csv'

    @property
    def stats_json(self) -> Path:
        return self.root / 'stats.json'

    @property
    def correction_dict_path(self) -> Path:
        return self.root / 'correction_dict.json'

    @property
    def checkpoint_file(self) -> Path:
        return self.root / 'checkpoint.json'

    @property
    def metrics_log(self) -> Path:
        return self.root / 'metrics_log.jsonl'

    @property
    def runs_csv(self) -> Path:
        return self.root / 'runs.csv'

    @property
    def lora_save_path(self) -> Path:
        return self.artifacts_dir / 'lora_weights'

    def setup(self):
        """إنشاء جميع المجلدات المطلوبة — Create all required directories"""
        for d in [self.logs_dir, self.exports_dir, self.cache_dir,
                  self.artifacts_dir, self.backups_dir, self.tensorboard_dir]:
            d.mkdir(parents=True, exist_ok=True)
        # إنشاء ملف التغذية الراجعة إذا لم يكن موجوداً
        if not self.feedback_csv.exists():
            pd.DataFrame(columns=['timestamp', 'image_id', 'original_text',
                                 'corrected_text', 'status']).to_csv(
                self.feedback_csv, index=False, encoding='utf-8-sig')
        print(f'✅ Directories ready at {self.root}')

    def setup_easyocr_symlink(self):
        """إنشاء رابط رمزي لمكتبة EasyOCR على Drive — Symlink EasyOCR to Drive"""
        if IN_COLAB:
            easyocr_dir = Path.home() / '.EasyOCR'
            drive_easyocr = Path('/content/drive/MyDrive/EasyOCR')
            if not easyocr_dir.exists() and drive_easyocr.exists():
                easyocr_dir.symlink_to(drive_easyocr)
                print('✅ EasyOCR symlinked to Drive')

    @classmethod
    def for_colab(cls):
        """إعدادات مثالية لبيئة كولاب — Optimal settings for Colab"""
        return cls(
            project_root='/content/handwriting-ocr',
            use_gpu=True,
            dpi=300,
            trocr_batch_size=8,  # أصغر لتوفير الذاكرة — smaller for memory
        )


# إنشاء كائن الإعدادات العام — Create global config instance
CFG = Config.for_colab() if IN_COLAB else Config()
CFG.setup()
CFG.setup_easyocr_symlink()
print(f'📂 Project root: {CFG.root}')
print(f'🗄️ Database: {CFG.db_path}')


In [ ]:
# 📝 نظام السجلات — Logging System

_LOG_FILE = CFG.logs_dir / f'ocr_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
_EVENTS = CFG.logs_dir / 'ocr_events.jsonl'

LOGGER = logging.getLogger('HandwrittenOCR')
LOGGER.setLevel(getattr(logging, CFG.log_level, logging.INFO))
LOGGER.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for _h in [RotatingFileHandler(_LOG_FILE, maxBytes=2_000_000, backupCount=5, encoding='utf-8'),
           logging.StreamHandler()]:
    _h.setFormatter(_fmt)
    LOGGER.addHandler(_h)

def log_event(event_type: str, payload: dict = None, level: str = 'info'):
    """تسجيل حدث بتنسيق JSONL — Log event in JSONL format"""
    payload = payload or {}
    entry = {'ts': datetime.now().isoformat(), 'event': event_type, 'data': payload}
    with open(_EVENTS, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    getattr(LOGGER, level, LOGGER.info)(f'{event_type} | {json.dumps(payload, ensure_ascii=False)}')

print('✅ Logging system ready')


In [ ]:
# 🗄️ قاعدة البيانات — Database System

SCHEMA_V3 = """
CREATE TABLE IF NOT EXISTS handwriting_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    page_num INTEGER NOT NULL,
    image_id TEXT NOT NULL,
    x INTEGER, y INTEGER, w INTEGER, h INTEGER,
    raw_text TEXT DEFAULT '',
    predicted_text TEXT DEFAULT '',
    corrected_text TEXT DEFAULT '',
    confidence REAL DEFAULT 0.0,
    ocr_engine TEXT DEFAULT 'ensemble',
    language TEXT DEFAULT 'unknown',
    status TEXT DEFAULT 'pending',
    is_low_confidence INTEGER DEFAULT 0,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS processing_runs (
    run_id TEXT PRIMARY KEY,
    input_path TEXT,
    start_page INTEGER,
    end_page INTEGER,
    total_pages INTEGER DEFAULT 0,
    total_words INTEGER DEFAULT 0,
    avg_confidence REAL DEFAULT 0.0,
    duration_sec REAL DEFAULT 0.0,
    status TEXT DEFAULT 'running',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS review_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT,
    image_id TEXT,
    original_text TEXT,
    corrected_text TEXT,
    action TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_status ON handwriting_data(status);
CREATE INDEX IF NOT EXISTS idx_page ON handwriting_data(page_num);
CREATE INDEX IF NOT EXISTS idx_run ON handwriting_data(run_id);
CREATE INDEX IF NOT EXISTS idx_conf ON handwriting_data(confidence);
"""


class HandwritingDB:
    """واجهة قاعدة البيانات — Database interface"""

    def __init__(self, db_path):
        self._db_path = Path(db_path)
        self._db_path.parent.mkdir(parents=True, exist_ok=True)
        self._create_tables()

    def _conn(self):
        conn = sqlite3.connect(str(self._db_path))
        conn.row_factory = sqlite3.Row
        conn.execute('PRAGMA journal_mode=WAL')
        conn.execute('PRAGMA foreign_keys=ON')
        return conn

    def _create_tables(self):
        with self._conn() as c:
            c.executescript(SCHEMA_V3)

    def _migrate(self):
        """ترقية قاعدة البيانات — Migrate database schema"""
        pass  # يمكن إضافة ترقيات مستقبلية هنا

    def insert_word(self, run_id, page_num, image_id, x, y, w, h,
                   raw_text, predicted_text, confidence, ocr_engine,
                   language='unknown', is_low_confidence=0):
        """إدراج كلمة جديدة — Insert a new word"""
        with self._conn() as c:
            c.execute("""
                INSERT INTO handwriting_data
                (run_id, page_num, image_id, x, y, w, h,
                 raw_text, predicted_text, confidence, ocr_engine,
                 language, is_low_confidence, status)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'pending')
            """, (run_id, page_num, image_id, x, y, w, h,
                  raw_text, predicted_text, confidence, ocr_engine,
                  language, is_low_confidence))
            return c.lastrowid

    def update_word(self, word_id, corrected_text, status='verified'):
        """تحديث كلمة — Update a word"""
        with self._conn() as c:
            c.execute("""
                UPDATE handwriting_data
                SET corrected_text=?, status=?, updated_at=CURRENT_TIMESTAMP
                WHERE id=?
            """, (corrected_text, status, word_id))

    def delete_word(self, word_id):
        """حذف كلمة — Delete a word"""
        with self._conn() as c:
            c.execute('DELETE FROM handwriting_data WHERE id=?', (word_id,))

    def delete_pages(self, run_id=None, page_num=None):
        """حذف صفحات — Delete pages"""
        with self._conn() as c:
            if run_id and page_num:
                c.execute('DELETE FROM handwriting_data WHERE run_id=? AND page_num=?',
                          (run_id, page_num))
            elif run_id:
                c.execute('DELETE FROM handwriting_data WHERE run_id=?', (run_id,))
            elif page_num:
                c.execute('DELETE FROM handwriting_data WHERE page_num=?', (page_num,))

    def get_all(self, run_id=None, page_num=None, status=None):
        """جلب جميع البيانات — Fetch all data"""
        q = 'SELECT * FROM handwriting_data WHERE 1=1'
        params = []
        if run_id:
            q += ' AND run_id=?'
            params.append(run_id)
        if page_num is not None:
            q += ' AND page_num=?'
            params.append(page_num)
        if status:
            q += ' AND status=?'
            params.append(status)
        q += ' ORDER BY page_num, y, x'
        with self._conn() as c:
            return [dict(r) for r in c.execute(q, params).fetchall()]

    def get_unverified(self, run_id=None):
        """جلب الكلمات غير المراجعة — Fetch unverified words"""
        return self.get_all(run_id=run_id, status='pending')

    def get_verified(self, run_id=None):
        """جلب الكلمات المراجعة — Fetch verified words"""
        return self.get_all(run_id=run_id, status='verified')

    def get_low_confidence(self, run_id=None, threshold=None):
        """جلب الكلمات منخفضة الثقة — Fetch low confidence words"""
        threshold = threshold or CFG.low_conf_threshold
        with self._conn() as c:
            q = 'SELECT * FROM handwriting_data WHERE confidence < ? AND is_low_confidence=1'
            params = [threshold]
            if run_id:
                q += ' AND run_id=?'
                params.append(run_id)
            q += ' ORDER BY confidence ASC'
            return [dict(r) for r in c.execute(q, params).fetchall()]

    def count_by_status(self, run_id=None):
        """عد الكلمات حسب الحالة — Count words by status"""
        with self._conn() as c:
            q = 'SELECT status, COUNT(*) as cnt FROM handwriting_data'
            params = []
            if run_id:
                q += ' WHERE run_id=?'
                params.append(run_id)
            q += ' GROUP BY status'
            return {r['status']: r['cnt'] for r in c.execute(q, params).fetchall()}

    def insert_run(self, run_id, input_path, start_page, end_page):
        """بدء تشغيل جديد — Start a new run"""
        with self._conn() as c:
            c.execute("""
                INSERT INTO processing_runs
                (run_id, input_path, start_page, end_page, status)
                VALUES (?, ?, ?, ?, 'running')
            """, (run_id, str(input_path), start_page, end_page))

    def finish_run(self, run_id, total_pages, total_words, avg_confidence, duration_sec):
        """إنهاء التشغيل — Finish a run"""
        with self._conn() as c:
            c.execute("""
                UPDATE processing_runs
                SET total_pages=?, total_words=?, avg_confidence=?,
                    duration_sec=?, status='completed'
                WHERE run_id=?
            """, (total_pages, total_words, avg_confidence, duration_sec, run_id))

    def log_review(self, run_id, image_id, original, corrected, action):
        """تسجيل مراجعة — Log a review event"""
        with self._conn() as c:
            c.execute("""
                INSERT INTO review_events
                (run_id, image_id, original_text, corrected_text, action)
                VALUES (?, ?, ?, ?, ?)
            """, (run_id, image_id, original, corrected, action))

# إنشاء قاعدة البيانات — Create database
DB = HandwritingDB(CFG.db_path)
print(f'✅ Database ready at {CFG.db_path}')


In [ ]:
# 🖼️ معالجة الصور — Image Preprocessing

def deskew(gray):
    """تصحيح ميل الصفحة باستخدام minAreaRect — Correct page skew"""
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3:
        return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                           borderMode=cv2.BORDER_REPLICATE)


def preprocess_image(img_bgr, adaptive=False):
    """مسار المعالجة الكامل: تصحيح الميل ← CLAHE ← إزالة الضوضاء ← العتبة"""
    """Full pipeline: deskew -> CLAHE -> denoise -> threshold"""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr.copy()
    if CFG.enable_deskew:
        gray = deskew(gray)
    gray = cv2.createCLAHE(clipLimit=CFG.clahe_clip,
                            tileGridSize=CFG.clahe_tile).apply(gray)
    gray = cv2.fastNlMeansDenoising(gray, h=CFG.denoise_h)
    if adaptive:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary, gray


def _iou(b1, b2):
    """حساب تقاطع الاتحاد — Compute Intersection over Union"""
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    xi1, yi1 = max(x1, x2), max(y1, y2)
    xi2, yi2 = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0


def crop_safe(img, x, y, w, h, pad=5):
    """قص آمن مع حشوة — Safe crop with padding"""
    H, W = img.shape[:2]
    return img[max(0, y - pad):min(H, y + h + pad),
               max(0, x - pad):min(W, x + w + pad)]


def smart_segmentation(img_bgr, binary, easyocr_detections=None):
    """تقسيم ذكي للكلمات: EasyOCR أولاً ثم الحدود البديلة"""
    """Smart word segmentation: EasyOCR first, then contour fallback"""
    H, W = binary.shape[:2]
    boxes = []

    # المرحلة الأولى: استخدام نتائج EasyOCR إذا وجدت
    if easyocr_detections:
        for det in easyocr_detections:
            pts = det[0]
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            x, y = int(min(xs)), int(min(ys))
            w, h = int(max(xs)) - x, int(max(ys)) - y
            if w > 5 and h > 5:
                boxes.append((x, y, w, h))

    # المرحلة الثانية: الحدود البديلة (Contours) إذا لم تكن هناك نتائج كافية
    if len(boxes) < 3:
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        min_area = (W * H) * 0.0002
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            aspect = w / max(h, 1)
            if w > 8 and h > 8 and w * h > min_area and aspect < 15:
                boxes.append((x, y, w, h))

    # إزالة التداخل بين الصناديق
    boxes = _merge_overlapping_boxes(boxes)
    return boxes


def _merge_overlapping_boxes(boxes, iou_threshold=0.3):
    """دمج الصناديق المتداخلة — Merge overlapping boxes"""
    if not boxes:
        return []
    keep = [True] * len(boxes)
    for i in range(len(boxes)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(boxes)):
            if not keep[j]:
                continue
            if _iou(boxes[i], boxes[j]) > iou_threshold:
                # دمج في الصندوق الأكبر
                x1, y1, w1, h1 = boxes[i]
                x2, y2, w2, h2 = boxes[j]
                nx = min(x1, x2)
                ny = min(y1, y2)
                nx2 = max(x1 + w1, x2 + w2)
                ny2 = max(y1 + h1, y2 + h2)
                boxes[i] = (nx, ny, nx2 - nx, ny2 - ny)
                keep[j] = False
    return [b for b, k in zip(boxes, keep) if k]


def match_boxes_with_detections(boxes, detections):
    """مطابقة الصناديق مع نتائج EasyOCR باستخدام IoU"""
    """Match boxes with EasyOCR detections using IoU"""
    mapping = {}
    used = set()
    for i, box in enumerate(boxes):
        best_j, best_iou = -1, 0.3
        for j, det in enumerate(detections):
            if j in used:
                continue
            pts = det[0]
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            dx = int(min(xs))
            dy = int(min(ys))
            dw = int(max(xs)) - dx
            dh = int(max(ys)) - dy
            iou_val = _iou(box, (dx, dy, dw, dh))
            if iou_val > best_iou:
                best_iou = iou_val
                best_j = j
        if best_j >= 0:
            mapping[i] = detections[best_j]
            used.add(best_j)
    return mapping

print('✅ Image preprocessing ready')


In [ ]:
# 🧠 محرك OCR — OCR Engine

%%time

# تحديد الجهاز — Determine device
DEVICE = torch.device('cuda' if torch.cuda.is_available() and CFG.use_gpu else 'cpu')
LOGGER.info(f'Device: {DEVICE}')

# تحميل TrOCR — Load TrOCR
_hf_kwargs = {}
if CFG.hf_token:
    _hf_kwargs['token'] = CFG.hf_token
if CFG.cache_dir.exists():
    _hf_kwargs['cache_dir'] = str(CFG.cache_dir)

trocr_processor = TrOCRProcessor.from_pretrained(CFG.trocr_model_name, **_hf_kwargs)
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    CFG.trocr_model_name, **_hf_kwargs
).to(DEVICE)

# تحميل أوزان LoRA إذا وجدت — Load LoRA weights if available
if CFG.lora_save_path.exists():
    from peft import PeftModel
    trocr_model = PeftModel.from_pretrained(trocr_model, str(CFG.lora_save_path)).to(DEVICE)
    trocr_model = trocr_model.merge_and_unload()
    print('✅ LoRA weights loaded and merged')

# تحميل EasyOCR مع العربية والإنجليزية والألمانية
easy_reader = easyocr.Reader(CFG.ocr_languages, gpu=torch.cuda.is_available())

# إعدادات Tesseract للنص المطبوع — Tesseract config for printed text
TESSERACT_LANG = CFG.tesseract_lang_config

# مدققي الإملاء — Spell checkers
_ar_corrector = Corrector() if _AR_CORRECTOR_AVAILABLE else None
_en_spellcheck = SpellChecker(language='en')
try:
    _de_spellcheck = SpellChecker(language='de')
except Exception:
    _de_spellcheck = None


def normalize_text(x):
    """تنظيف النص — Clean text"""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ''
    return str(x).strip()


def detect_lang(text):
    """كشف اللغة — Detect language"""
    try:
        return detect(str(text)) if text and text.strip() else 'unknown'
    except Exception:
        return 'unknown'


def spell_correct(text):
    """تصحيح الإملاء حسب اللغة — Spell check based on language"""
    text = normalize_text(text)
    if not text:
        return ''
    try:
        lang = detect_lang(text)
        if lang == 'ar' and _ar_corrector:
            return _ar_corrector.contextual_correct(text)
        elif lang == 'de' and _de_spellcheck:
            words = text.split()
            return ' '.join(_de_spellcheck.correction(w) or w for w in words)
        else:
            words = text.split()
            return ' '.join(_en_spellcheck.correction(w) or w for w in words)
    except Exception:
        return text


def tesseract_ocr(crop):
    """استخدام Tesseract للنص المطبوع — Use Tesseract for printed text"""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
    text = pytesseract.image_to_string(
        gray, lang=TESSERACT_LANG, config='--psm 6'
    ).strip()
    return text


def trocr_batch_predict(crops):
    """استنتاج TrOCR دفعة واحدة — Batch TrOCR inference (3-6x speedup)"""
    if not crops:
        return []
    pil_imgs = []
    for crop in crops:
        if crop.ndim == 3:
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        else:
            rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        pil_imgs.append(Image.fromarray(rgb))
    try:
        pv = trocr_processor(
            images=pil_imgs, return_tensors='pt', padding=True
        ).pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(
                pv, max_length=CFG.max_text_length,
                num_beams=CFG.num_beams,
                early_stopping=True,
                length_penalty=1.0
            )
        return trocr_processor.batch_decode(ids, skip_special_tokens=True)
    except Exception as e:
        LOGGER.warning(f'trocr_batch failed: {e}')
        return [''] * len(crops)


def trocr_single_predict(crop):
    """استنتاج TrOCR لكلمة واحدة — Single TrOCR inference"""
    try:
        if crop.ndim == 3:
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        else:
            rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        pv = trocr_processor(
            images=Image.fromarray(rgb), return_tensors='pt'
        ).pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(
                pv, max_length=CFG.max_text_length,
                num_beams=CFG.num_beams,
                early_stopping=True
            )
        txt = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        return txt
    except Exception as e:
        LOGGER.debug(f'trocr_single: {e}')
        return ''


def ocr_ensemble_single(crop, easy_item=None):
    """التصويت الذكي: تخطي TrOCR إذا كانت ثقة EasyOCR عالية"""
    """Smart ensemble: skip TrOCR if EasyOCR confidence > threshold"""
    candidates = []

    # EasyOCR أولاً — EasyOCR first
    if easy_item is not None:
        _, txt, conf = easy_item
        txt = normalize_text(txt)
        if txt:
            candidates.append(('easyocr', txt, float(conf)))
            # تخطي TrOCR إذا كانت الثقة عالية — Skip TrOCR if high confidence
            if float(conf) >= CFG.easy_conf_threshold:
                return txt, float(conf), 'easyocr', False, candidates

    # Tesseract للنص المطبوع — Tesseract for printed text
    try:
        tess_text = tesseract_ocr(crop)
        if tess_text:
            candidates.append(('tesseract', tess_text, 0.75))
    except Exception:
        pass

    # TrOCR فقط عند الحاجة — TrOCR only when needed
    try:
        txt = trocr_single_predict(crop)
        if txt:
            candidates.append(('trocr', txt, CFG.trocr_default_conf))
    except Exception as e:
        LOGGER.debug(f'trocr_single: {e}')

    candidates = [c for c in candidates if c[1]]
    if not candidates:
        return '', 0.0, 'none', True, []

    best = max(candidates, key=lambda x: x[2])
    return best[1], float(best[2]), best[0], best[2] < CFG.low_conf_threshold, candidates


print(f'✅ OCR Engine ready on {DEVICE}')


In [ ]:
# 📖 نظام القاموس التصحيحي — Correction Dictionary System

def build_correction_dict():
    """بناء قاموس التصحيح التلقائي من التغذية الراجعة"""
    """Build auto-correction dict from user feedback"""
    if not CFG.feedback_csv.exists():
        return {}
    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty:
            return {}
        buckets = defaultdict(Counter)
        for _, row in df.iterrows():
            o = normalize_text(row.get('original_text'))
            c = normalize_text(row.get('corrected_text'))
            if o and c and o != c:
                buckets[o][c] += 1
        result = {}
        for o, cnt in buckets.items():
            top = cnt.most_common(1)[0]
            if top[1] >= CFG.correction_min_votes:
                result[o] = top[0]
        CFG.correction_dict_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2), 'utf-8'
        )
        log_event('correction_dict_built', {'entries': len(result)})
        return result
    except Exception as e:
        LOGGER.error(f'correction_dict: {e}')
        return {}


def load_correction_dict():
    """تحميل القاموس التصحيحي — Load correction dictionary"""
    if CFG.correction_dict_path.exists():
        return json.loads(CFG.correction_dict_path.read_text('utf-8'))
    return build_correction_dict()


def apply_corrections(text, d):
    """تطبيق التصحيحات على النص — Apply corrections to text"""
    if not text:
        return ''
    return ' '.join(d.get(tok, tok) for tok in text.split())


def append_feedback(image_id, original, corrected, status='verified'):
    """إضافة تغذية راجعة — Append feedback entry"""
    ts = datetime.now().isoformat()
    pd.DataFrame([{'timestamp': ts, 'image_id': image_id,
                   'original_text': original, 'corrected_text': corrected,
                   'status': status}]).to_csv(
        CFG.feedback_csv, mode='a', header=not CFG.feedback_csv.exists(),
        index=False, encoding='utf-8-sig'
    )


# بناء القاموس عند البدء — Build dictionary on startup
correction_dict = load_correction_dict()
print(f'✅ Correction dict: {len(correction_dict)} entries')


In [ ]:
# 💾 نظام نقاط التفتيش — Checkpoint System

def save_checkpoint(data):
    """حفظ نقطة تفتيش — Save checkpoint"""
    CFG.checkpoint_file.write_text(
        json.dumps(data, ensure_ascii=False, indent=2), 'utf-8'
    )
    LOGGER.info(f'Checkpoint saved: page {data.get("current_page")}')


def load_checkpoint():
    """تحميل نقطة تفتيش — Load checkpoint"""
    if CFG.checkpoint_file.exists():
        data = json.loads(CFG.checkpoint_file.read_text('utf-8'))
        LOGGER.info(f'Checkpoint loaded: page {data.get("current_page")}')
        return data
    return None


def clear_checkpoint():
    """مسح نقطة التفتيش — Clear checkpoint"""
    if CFG.checkpoint_file.exists():
        CFG.checkpoint_file.unlink()
        LOGGER.info('Checkpoint cleared')


print('✅ Checkpoint system ready')


In [ ]:
# 📄 معالجة المستندات — PDF/Image Processing

def load_pages(input_path, start_page=None, end_page=None):
    """تحميل الصفحات من PDF أو صورة — Load pages from PDF or image"""
    input_path = str(input_path)
    ext = Path(input_path).suffix.lower()
    sp = start_page or CFG.pages_start
    ep = end_page or CFG.pages_end
    if ext == '.pdf':
        LOGGER.info(f'Loading PDF: {input_path}, pages {sp}-{ep or "all"}')
        kwargs = {'dpi': CFG.dpi, 'first_page': sp}
        if ep:
            kwargs['last_page'] = ep
        imgs = convert_from_path(input_path, **kwargs)
        pages = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in imgs]
        return pages, list(range(sp, sp + len(pages)))
    if ext in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}:
        LOGGER.info(f'Loading image: {input_path}')
        img = cv2.imread(input_path)
        if img is None:
            raise ValueError(f'Cannot read image: {input_path}')
        return [img], [1]
    raise ValueError(f'Unsupported format: {ext}')


def process_document(input_path=None, start_page=None, end_page=None,
                     resume=True, adaptive=False, progress_cb=None):
    """المعالجة الرئيسية مع استئناف نقطة التفتيش، TrOCR دفعي، تصدير تلقائي"""
    """Main processing with checkpoint resume, batch TrOCR, auto-export"""
    input_path = input_path or CFG.pdf_path
    if not input_path:
        return {'error': 'No input path provided'}

    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    start_time = time.time()
    sp = start_page or CFG.pages_start
    ep = end_page or CFG.pages_end

    # استئناف من نقطة تفتيش — Resume from checkpoint
    ckpt = load_checkpoint() if resume else None
    if ckpt:
        run_id = ckpt.get('run_id', run_id)
        start_time = ckpt.get('start_time', start_time)
        sp = ckpt.get('next_page', sp)
        LOGGER.info(f'Resuming from page {sp}')

    # تسجيل التشغيل — Register run
    DB.insert_run(run_id, input_path, sp, ep)

    try:
        # تحميل الصفحات — Load pages
        pages, page_nums = load_pages(input_path, sp, ep)
        total_words = 0
        all_confidences = []

        for page_idx, (page_img, page_num) in enumerate(zip(pages, page_nums)):
            LOGGER.info(f'Processing page {page_num} ({page_idx + 1}/{len(pages)})')
            if progress_cb:
                progress_cb(page_idx + 1, len(pages), f'Page {page_num}')

            # معالجة الصورة — Preprocess
            binary, gray = preprocess_image(page_img, adaptive=adaptive)

            # كشف النصوص باستخدام EasyOCR — Detect text with EasyOCR
            try:
                easy_dets = easy_reader.readtext(page_img)
            except Exception as e:
                LOGGER.warning(f'EasyOCR failed on page {page_num}: {e}')
                easy_dets = []

            # تقسيم الكلمات — Segment words
            boxes = smart_segmentation(page_img, binary, easy_dets)
            det_map = match_boxes_with_detections(boxes, easy_dets)

            # تجميع الكلمات ذات الثقة المنخفضة لـ TrOCR الدفعي
            low_conf_crops = []
            low_conf_indices = []
            page_word_count = 0

            for i, (x, y, w, h) in enumerate(boxes):
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0:
                    continue

                image_id = f'p{page_num}_w{i}'
                easy_item = det_map.get(i)

                # التحقق مما إذا كان يمكن تخطي TrOCR — Check if TrOCR can be skipped
                need_trocr = True
                if easy_item is not None:
                    _, txt, conf = easy_item
                    if float(conf) >= CFG.easy_conf_threshold:
                        need_trocr = False

                if need_trocr:
                    low_conf_crops.append(crop)
                    low_conf_indices.append(i)
                else:
                    # معالجة مباشرة بدون TrOCR — Direct processing without TrOCR
                    _, txt, conf = easy_item
                    txt = normalize_text(txt)
                    lang = detect_lang(txt) if txt else 'unknown'
                    corrected = apply_corrections(txt, correction_dict) if txt else ''
                    DB.insert_word(
                        run_id, page_num, image_id, x, y, w, h,
                        txt, corrected, float(conf), 'easyocr', lang, 0
                    )
                    all_confidences.append(float(conf))
                    page_word_count += 1

            # TrOCR دفعي للكلمات منخفضة الثقة — Batch TrOCR for low confidence
            if low_conf_crops:
                batch_texts = trocr_batch_predict(low_conf_crops)
                for idx, (i, txt) in enumerate(zip(low_conf_indices, batch_texts)):
                    x, y, w, h = boxes[i]
                    image_id = f'p{page_num}_w{i}'
                    txt = normalize_text(txt)
                    easy_item = det_map.get(i)

                    # مقارنة مع EasyOCR — Compare with EasyOCR
                    candidates = []
                    if easy_item is not None:
                        _, e_txt, e_conf = easy_item
                        e_txt = normalize_text(e_txt)
                        if e_txt:
                            candidates.append(('easyocr', e_txt, float(e_conf)))
                    if txt:
                        candidates.append(('trocr', txt, CFG.trocr_default_conf))
                    try:
                        tess_text = tesseract_ocr(crop_safe(page_img, x, y, w, h))
                        if tess_text:
                            candidates.append(('tesseract', tess_text, 0.75))
                    except Exception:
                        pass

                    candidates = [c for c in candidates if c[1]]
                    if candidates:
                        best = max(candidates, key=lambda c: c[2])
                        final_text = best[1]
                        final_conf = float(best[2])
                        engine = best[0]
                    else:
                        final_text = ''
                        final_conf = 0.0
                        engine = 'none'

                    lang = detect_lang(final_text) if final_text else 'unknown'
                    corrected = apply_corrections(final_text, correction_dict) if final_text else ''
                    is_low = 1 if final_conf < CFG.low_conf_threshold else 0

                    DB.insert_word(
                        run_id, page_num, image_id, x, y, w, h,
                        final_text, corrected, final_conf, engine, lang, is_low
                    )
                    all_confidences.append(final_conf)
                    page_word_count += 1

            total_words += page_word_count

            # حفظ نقطة تفتيش بعد كل صفحة — Save checkpoint after each page
            save_checkpoint({
                'run_id': run_id,
                'start_time': start_time,
                'next_page': page_num + 1,
                'total_words': total_words,
            })

            # تنظيف ذاكرة GPU — GPU memory cleanup
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # إنهاء التشغيل — Finish run
        duration = time.time() - start_time
        avg_conf = np.mean(all_confidences) if all_confidences else 0.0
        DB.finish_run(run_id, len(pages), total_words, avg_conf, duration)

        # مسح نقطة التفتيش — Clear checkpoint
        clear_checkpoint()

        # تصدير تلقائي — Auto export
        if CFG.auto_export_after_run:
            try:
                auto_export(run_id)
            except Exception as e:
                LOGGER.error(f'Auto-export failed: {e}')

        # تسجيل الحدث — Log event
        log_event('processing_complete', {
            'run_id': run_id, 'pages': len(pages), 'words': total_words,
            'avg_confidence': round(avg_conf, 4), 'duration_sec': round(duration, 1)
        })

        return {
            'run_id': run_id, 'pages': len(pages), 'words': total_words,
            'avg_confidence': avg_conf, 'duration_sec': duration,
            'status': 'success'
        }

    except Exception as e:
        LOGGER.error(f'Processing failed: {e}', exc_info=True)
        return {'error': str(e), 'run_id': run_id}


print('✅ PDF/Image processor ready')


In [ ]:
# 📤 نظام التصدير التلقائي — Auto-Export System

def auto_export(run_id, output_dir=None):
    """تصدير تلقائي: CSV + XLSX + نص كامل + JSONL بيانات تدريب"""
    """Auto export: CSV + XLSX + full text + JSONL training data"""
    output_dir = Path(output_dir) if output_dir else CFG.exports_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    # جلب البيانات — Fetch data
    data = DB.get_all(run_id=run_id)
    if not data:
        LOGGER.warning('No data to export')
        return

    df = pd.DataFrame(data)

    # 1. تصدير CSV — Export CSV
    csv_path = output_dir / f'ocr_results_{ts}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    LOGGER.info(f'CSV exported: {csv_path}')

    # 2. تصدير XLSX — Export XLSX
    try:
        xlsx_path = output_dir / f'ocr_results_{ts}.xlsx'
        df.to_excel(xlsx_path, index=False, engine='openpyxl')
        LOGGER.info(f'XLSX exported: {xlsx_path}')
    except Exception as e:
        LOGGER.warning(f'XLSX export failed: {e}')

    # 3. تصدير نص كامل — Export full text
    txt_path = output_dir / f'ocr_fulltext_{ts}.txt'
    sentences = reconstruct_sentences(run_id=run_id, verified_only=False)
    with open(txt_path, 'w', encoding='utf-8') as f:
        for page_num, sent in sentences:
            f.write(f'--- Page {page_num} ---\n')
            f.write(sent + '\n\n')
    LOGGER.info(f'Full text exported: {txt_path}')

    # 4. تصدير بيانات التدريب JSONL — Export training data JSONL
    verified = DB.get_verified(run_id=run_id)
    if verified:
        jsonl_path = output_dir / f'training_data_{ts}.jsonl'
        with open(jsonl_path, 'w', encoding='utf-8') as f:
            for row in verified:
                entry = {
                    'image_id': row['image_id'],
                    'raw_text': row['raw_text'],
                    'corrected_text': row['corrected_text'],
                    'page_num': row['page_num'],
                    'confidence': row['confidence'],
                }
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        LOGGER.info(f'Training data exported: {jsonl_path}')

    # 5. تحديث الإحصائيات — Update stats
    stats = {
        'last_run_id': run_id,
        'last_export': ts,
        'total_words': len(data),
        'verified_words': len(DB.get_verified(run_id=run_id)),
        'avg_confidence': float(df['confidence'].mean()) if 'confidence' in df.columns else 0,
    }
    CFG.stats_json.write_text(json.dumps(stats, indent=2, ensure_ascii=False), 'utf-8')

    log_event('auto_export', {'run_id': run_id, 'files': len(list(output_dir.glob(f'*{ts}*')))})
    return str(output_dir)


def create_backup():
    """إنشاء نسخة احتياطية من قاعدة البيانات والملفات — Backup DB and artifacts"""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_dir = CFG.backups_dir / ts
    backup_dir.mkdir(parents=True, exist_ok=True)
    # نسخ قاعدة البيانات — Copy database
    if CFG.db_path.exists():
        shutil.copy2(CFG.db_path, backup_dir / CFG.db_name)
    # نسخ ملف التغذية الراجعة — Copy feedback
    if CFG.feedback_csv.exists():
        shutil.copy2(CFG.feedback_csv, backup_dir / 'feedback.csv')
    # نسخ القاموس التصحيحي — Copy correction dict
    if CFG.correction_dict_path.exists():
        shutil.copy2(CFG.correction_dict_path, backup_dir / 'correction_dict.json')
    log_event('backup_created', {'dir': str(backup_dir)})
    LOGGER.info(f'Backup created: {backup_dir}')
    return str(backup_dir)


print('✅ Export system ready')


In [ ]:
# 🔗 إعادة بناء الجمل — Sentence Reconstruction

def reconstruct_sentences(verified_only=False, run_id=None, y_tolerance=25):
    """إعادة بناء الجمل من بيانات الكلمات مع دعم RTL للعربية"""
    """Reconstruct sentences from word-level data with RTL support"""
    status = 'verified' if verified_only else None
    data = DB.get_all(run_id=run_id, status=status)
    if not data:
        return []

    # تجميع حسب الصفحة — Group by page
    pages = defaultdict(list)
    for row in data:
        pages[row['page_num']].append(row)

    results = []
    for page_num in sorted(pages.keys()):
        words = sorted(pages[page_num], key=lambda w: (w.get('y', 0), w.get('x', 0)))

        # تجميع الكلمات في أسطر — Group words into lines
        lines = []
        current_line = []
        last_y = None

        for w in words:
            wy = w.get('y', 0)
            text = w.get('corrected_text') or w.get('raw_text') or ''
            if not text:
                continue
            if last_y is not None and abs(wy - last_y) > y_tolerance:
                if current_line:
                    lines.append(current_line)
                current_line = []
            current_line.append(w)
            last_y = wy
        if current_line:
            lines.append(current_line)

        # بناء النص لكل سطر — Build text for each line
        page_text = []
        for line in lines:
            if not line:
                continue
            # كشف اللغة — Detect language
            sample_text = ' '.join(
                (w.get('corrected_text') or w.get('raw_text') or '') for w in line
            )
            lang = detect_lang(sample_text)

            # ترتيب حسب اللغة — Sort by language
            sorted_line = sorted(line, key=lambda w: w.get('x', 0))
            line_words = [
                w.get('corrected_text') or w.get('raw_text') or ''
                for w in sorted_line
            ]
            line_text = ' '.join(w for w in line_words if w)

            # إعادة ترتيب الكلمات العربية (RTL) — Reorder Arabic words (RTL)
            if lang == 'ar':
                try:
                    import arabic_reshaper
                    from bidi.algorithm import get_display
                    reshaped = arabic_reshaper.reshape(line_text)
                    line_text = get_display(reshaped)
                except ImportError:
                    pass  # المكتبات غير متوفرة — libraries not available

            page_text.append(line_text)

        full_page = '\n'.join(page_text)
        results.append((page_num, full_page))

    return results


print('✅ Sentence reconstruction ready')


In [ ]:
# 📊 نظام المقاييس — WER/CER Metrics

def compute_metrics(run_id=None):
    """حساب WER و CER باستخدام jiwer — Compute WER and CER"""
    try:
        from jiwer import wer, cer
    except ImportError:
        return {'error': 'pip install jiwer'}

    # جلب البيانات المراجعة — Fetch verified data
    verified = DB.get_verified(run_id=run_id)
    if not verified:
        return {'message': 'No verified data available for metrics'}

    wers = []
    cers = []
    for row in verified:
        raw = normalize_text(row.get('raw_text', ''))
        corrected = normalize_text(row.get('corrected_text', ''))
        if raw and corrected:
            wers.append(wer(raw, corrected))
            cers.append(cer(raw, corrected))

    result = {}
    if wers:
        result['wer_mean'] = float(np.mean(wers))
        result['wer_median'] = float(np.median(wers))
        result['cer_mean'] = float(np.mean(cers))
        result['cer_median'] = float(np.median(cers))
        result['num_samples'] = len(wers)

    # حفظ المقاييس — Save metrics
    entry = {
        'ts': datetime.now().isoformat(),
        'run_id': run_id or 'all',
        **result
    }
    with open(CFG.metrics_log, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    log_event('metrics_computed', result)
    return result


def plot_metrics_history():
    """رسم WER/CER عبر الوقت — Plot WER/CER improvement over time"""
    if not CFG.metrics_log.exists():
        print('No metrics history found')
        return None

    records = []
    with open(CFG.metrics_log, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    if not records:
        print('No metrics records found')
        return None

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # WER plot
    wers = [r.get('wer_mean', 0) for r in records]
    ts = [r.get('ts', '')[:16] for r in records]
    axes[0].plot(range(len(wers)), wers, 'b-o', linewidth=2, markersize=8)
    axes[0].set_title('WER عبر الوقت — WER Over Time', fontsize=14)
    axes[0].set_xlabel('القياس — Measurement')
    axes[0].set_ylabel('WER')
    axes[0].grid(True, alpha=0.3)
    if len(ts) <= 10:
        axes[0].set_xticks(range(len(ts)))
        axes[0].set_xticklabels(ts, rotation=45, fontsize=8)

    # CER plot
    cers = [r.get('cer_mean', 0) for r in records]
    axes[1].plot(range(len(cers)), cers, 'r-s', linewidth=2, markersize=8)
    axes[1].set_title('CER عبر الوقت — CER Over Time', fontsize=14)
    axes[1].set_xlabel('القياس — Measurement')
    axes[1].set_ylabel('CER')
    axes[1].grid(True, alpha=0.3)
    if len(ts) <= 10:
        axes[1].set_xticks(range(len(ts)))
        axes[1].set_xticklabels(ts, rotation=45, fontsize=8)

    plt.tight_layout()
    plot_path = CFG.exports_dir / 'metrics_history.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Plot saved: {plot_path}')
    return str(plot_path)


def plot_confidence_distribution(run_id=None):
    """رسم توزيع الثقة — Plot confidence distribution"""
    data = DB.get_all(run_id=run_id)
    if not data:
        print('No data available')
        return None

    confidences = [r.get('confidence', 0) for r in data]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(confidences, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(x=CFG.low_conf_threshold, color='red', linestyle='--',
               label=f'Low conf threshold ({CFG.low_conf_threshold})')
    ax.axvline(x=CFG.easy_conf_threshold, color='green', linestyle='--',
               label=f'EasyOCR threshold ({CFG.easy_conf_threshold})')
    ax.set_title('توزيع الثقة — Confidence Distribution', fontsize=14)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = CFG.exports_dir / 'confidence_distribution.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    return str(plot_path)


print('✅ Metrics system ready')


In [ ]:
# 🎯 ضبط دقيق LoRA — LoRA Fine-tuning

from peft import get_peft_model, LoraConfig, TaskType
from torch.optim import AdamW
from torch.utils.data import Dataset as TorchDataset, DataLoader


class OCRDataset(TorchDataset):
    """مجموعة بيانات التدريب — Training dataset"""

    def __init__(self, db, run_id=None, processor=None):
        self.processor = processor or trocr_processor
        verified = db.get_verified(run_id=run_id)
        self.samples = []
        for row in verified:
            img_id = row.get('image_id', '')
            txt = normalize_text(row.get('corrected_text', ''))
            if txt and len(txt) <= CFG.max_text_length:
                self.samples.append({'image_id': img_id, 'text': txt})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        # إنشاء صورة وهمية من النص (للحالات التي لا توجد فيها صور فعلية)
        # في بيئة حقيقية، يتم تحميل الصورة من image_id
        dummy_img = np.zeros((48, 256, 3), dtype=np.uint8)
        pil_img = Image.fromarray(dummy_img)
        encoding = self.processor(
            images=pil_img, text=sample['text'],
            return_tensors='pt', padding=True
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding['labels'] = encoding['labels']
        return encoding


def finetune_trocr_lora(min_samples=None):
    """ضبط TrOCR مع LoRA على التصحيحات المراجعة"""
    """Fine-tune TrOCR with LoRA on verified user corrections"""
    global trocr_model

    min_samples = min_samples or CFG.finetune_min_samples
    verified = DB.get_verified()
    if len(verified) < min_samples:
        msg = f'Need at least {min_samples} verified samples, have {len(verified)}'
        LOGGER.warning(msg)
        return {'error': msg}

    LOGGER.info(f'Starting LoRA fine-tuning with {len(verified)} samples')

    # إعداد LoRA — Configure LoRA
    lora_config = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        target_modules=['q_proj', 'v_proj'],
        bias='none',
        task_type=TaskType.SEQ_2_SEQ_LM,
    )

    # تطبيق LoRA — Apply LoRA
    try:
        model_to_train = get_peft_model(trocr_model, lora_config)
    except Exception as e:
        LOGGER.error(f'LoRA setup failed: {e}')
        return {'error': str(e)}

    model_to_train.print_trainable_parameters()
    model_to_train.to(DEVICE)

    # إعداد مجموعة البيانات — Setup dataset
    dataset = OCRDataset(DB, processor=trocr_processor)
    if len(dataset) == 0:
        return {'error': 'No valid training samples after filtering'}

    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    optimizer = AdamW(model_to_train.parameters(), lr=CFG.learning_rate)

    # حلقة التدريب — Training loop
    model_to_train.train()
    losses = []
    for epoch in range(CFG.finetune_epochs):
        epoch_loss = 0.0
        num_batches = 0
        for batch in dataloader:
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model_to_train(
                pixel_values=pixel_values,
                labels=labels
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / max(num_batches, 1)
        losses.append(avg_loss)
        LOGGER.info(f'Epoch {epoch + 1}/{CFG.finetune_epochs}, Loss: {avg_loss:.4f}')

        # تنظيف الذاكرة — Memory cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # حفظ أوزان LoRA — Save LoRA weights
    CFG.lora_save_path.mkdir(parents=True, exist_ok=True)
    model_to_train.save_pretrained(str(CFG.lora_save_path))
    LOGGER.info(f'LoRA weights saved to {CFG.lora_save_path}')

    # تحديث النموذج العام — Update global model
    trocr_model = PeftModel.from_pretrained(
        trocr_model.base_model if hasattr(trocr_model, 'base_model') else trocr_model,
        str(CFG.lora_save_path)
    ).to(DEVICE)

    # حفظ مقاييس التدريب — Save training metrics
    metrics = {
        'ts': datetime.now().isoformat(),
        'epochs': CFG.finetune_epochs,
        'samples': len(dataset),
        'final_loss': losses[-1] if losses else None,
        'all_losses': losses,
    }
    log_event('finetune_complete', metrics)

    return {
        'status': 'success',
        'epochs': CFG.finetune_epochs,
        'samples_trained': len(dataset),
        'final_loss': losses[-1] if losses else None,
    }


print('✅ Fine-tuning system ready')


In [ ]:
# 🔄 التعلم النشط — Active Learning

def check_active_learning_trigger():
    """التحقق من كفاية التصحيحات الجديدة لإعادة المعالجة"""
    """Check if enough new corrections accumulated to trigger reprocessing"""
    if not CFG.feedback_csv.exists():
        return {'trigger': False, 'message': 'No feedback file'}

    try:
        df = pd.read_csv(CFG.feedback_csv, encoding='utf-8-sig')
        if df.empty:
            return {'trigger': False, 'message': 'No feedback entries'}

        total_corrections = len(df)
        recent = df.tail(CFG.al_min_new_corrections)
        changed = recent[recent['original_text'] != recent['corrected_text']]

        should_trigger = len(changed) >= CFG.al_min_new_corrections
        msg = (f'{len(changed)} new corrections (threshold: {CFG.al_min_new_corrections}). '
               f'Retraining recommended.' if should_trigger else 'Not enough yet.')

        return {
            'trigger': should_trigger,
            'total_corrections': total_corrections,
            'recent_corrections': len(changed),
            'threshold': CFG.al_min_new_corrections,
            'message': msg
        }
    except Exception as e:
        return {'trigger': False, 'error': str(e)}


def reprocess_low_confidence(limit=100, run_id=None):
    """إعادة معالجة الكلمات منخفضة الثقة بالنموذج المحدث"""
    """Re-process low confidence words with updated model"""
    low_conf = DB.get_low_confidence(run_id=run_id)
    if not low_conf:
        return {'message': 'No low confidence words to reprocess'}

    updated = 0
    improved = 0
    for row in low_conf[:limit]:
        word_id = row['id']
        old_text = normalize_text(row.get('raw_text', ''))
        old_conf = row.get('confidence', 0.0)

        # محاولة إعادة OCR — Try re-OCR
        # في بيئة حقيقية، نعيد تحميل الصورة من image_id
        # هنا نستخدم TrOCR مباشرة على النص كحل بديل
        try:
            new_text = spell_correct(old_text)
            new_text = apply_corrections(new_text, correction_dict)
            if new_text and new_text != old_text:
                DB.update_word(word_id, new_text, 'auto_corrected')
                updated += 1
                if new_text != old_text:
                    improved += 1
        except Exception as e:
            LOGGER.debug(f'Reprocess failed for word {word_id}: {e}')

    log_event('active_learning_reprocess', {
        'checked': min(limit, len(low_conf)),
        'updated': updated,
        'improved': improved
    })

    return {
        'checked': min(limit, len(low_conf)),
        'updated': updated,
        'improved': improved
    }


print('✅ Active Learning ready')


In [ ]:
# 🤗 رفع إلى HuggingFace — HuggingFace Upload

from huggingface_hub import HfApi, login, create_repo


def upload_to_hf(repo_id=None, hf_token=None):
    """رفع مجموعة بيانات التدريب إلى HuggingFace Hub"""
    """Upload training dataset to HuggingFace Hub"""
    repo_id = repo_id or CFG.hf_dataset_repo
    hf_token = hf_token or CFG.hf_token or _HF_TOKEN

    if not hf_token:
        return '❌ HF_TOKEN required. Set CFG.hf_token or add to Colab Secrets.'
    if not repo_id:
        return '❌ repo_id required. Set CFG.hf_dataset_repo.'

    try:
        # تسجيل الدخول — Login
        login(token=hf_token)
        api = HfApi()

        # إنشاء المستودع إذا لم يكن موجوداً — Create repo if not exists
        try:
            create_repo(repo_id=repo_id, repo_type='dataset', exist_ok=True)
        except Exception:
            pass

        # رفع الملفات — Upload files
        uploaded = []
        for f in CFG.exports_dir.iterdir():
            if f.is_file() and f.suffix in {'.csv', '.jsonl', '.txt', '.json'}:
                try:
                    api.upload_file(
                        path_or_fileobj=str(f),
                        path_in_repo=f.name,
                        repo_id=repo_id,
                        repo_type='dataset',
                    )
                    uploaded.append(f.name)
                except Exception as e:
                    LOGGER.warning(f'Upload failed for {f.name}: {e}')

        url = f'https://huggingface.co/datasets/{repo_id}'
        log_event('hf_upload', {'repo': repo_id, 'files': len(uploaded)})
        return f'✅ Uploaded {len(uploaded)} files to {url}'

    except Exception as e:
        LOGGER.error(f'HF upload failed: {e}')
        return f'❌ Upload failed: {e}'


print('✅ HuggingFace upload ready')


In [ ]:
# 📚 استخراج المفردات ثنائية اللغة — Bilingual Vocabulary Extraction

def extract_bilingual_vocab(run_id=None):
    """استخراج أزواج المفردات من البيانات المراجعة"""
    """Extract vocabulary pairs from verified data"""
    verified = DB.get_verified(run_id=run_id)
    if not verified:
        return {}, 'No verified data available'

    vocab_by_lang = defaultdict(list)
    for row in verified:
        text = normalize_text(row.get('corrected_text') or row.get('raw_text', ''))
        if not text:
            continue
        lang = row.get('language', 'unknown') or detect_lang(text)
        for word in text.split():
            clean = word.strip('.,;:!?()[]{}"\'-')
            if clean and len(clean) > 1:
                vocab_by_lang[lang].append(clean.lower())

    # إحصائيات المفردات — Vocabulary statistics
    stats = {}
    for lang, words in vocab_by_lang.items():
        counter = Counter(words)
        stats[lang] = {
            'total_unique': len(counter),
            'total_occurrences': sum(counter.values()),
            'top_20': counter.most_common(20),
        }

    # حفظ النتائج — Save results
    output = {}
    for lang, data in stats.items():
        output[lang] = {
            'unique_words': data['total_unique'],
            'occurrences': data['total_occurrences'],
            'top_words': {w: c for w, c in data['top_20']},
        }

    vocab_path = CFG.exports_dir / 'vocabulary_stats.json'
    vocab_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), 'utf-8')

    log_event('vocab_extracted', {
        'languages': list(stats.keys()),
        'total_unique': sum(d['total_unique'] for d in stats.values())
    })

    # عرض ملخص — Show summary
    summary = f'Extracted vocabulary from {len(verified)} verified words\n'
    for lang, data in stats.items():
        summary += f'  {lang}: {data["total_unique"]} unique words, {data["total_occurrences"]} occurrences\n'
        summary += f'    Top: {", ".join(w for w, _ in data["top_20"][:10])}\n'

    return output, summary


print('✅ Vocabulary extraction ready')


In [ ]:
# 🚀 تشغيل المعالجة — Run Processing

# المسار الافتراضي — Default path (غيّره حسب ملفك — change to your file)
DEFAULT_PDF = '/content/sample_document.pdf'  # ← غيّر هذا المسار

# إذا لم يكن الملف موجوداً، نستخدم ملف تجريبي — Use demo if no file
if not Path(DEFAULT_PDF).exists():
    # إنشاء صورة تجريبية — Create a demo image
    demo_img = np.ones((800, 600, 3), dtype=np.uint8) * 255
    cv2.putText(demo_img, 'Hello World / مرحبا بالعالم', (50, 400),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
    demo_path = CFG.root / 'demo_page.png'
    cv2.imwrite(str(demo_path), demo_img)
    DEFAULT_PDF = str(demo_path)
    print(f'📄 Created demo image: {demo_path}')

result = process_document(
    input_path=DEFAULT_PDF,
    start_page=1,
    end_page=None,  # None = كل الصفحات — all pages
    resume=True,
    adaptive=False
)

if 'error' in result:
    print(f'❌ Error: {result["error"]}')
else:
    print(f'\n📊 Processing Summary:')
    print(f'   Run ID: {result.get("run_id", "N/A")}')
    print(f'   Pages: {result.get("pages", 0)}')
    print(f'   Words: {result.get("words", 0)}')
    print(f'   Avg Confidence: {result.get("avg_confidence", 0):.2%}')
    print(f'   Duration: {result.get("duration_sec", 0):.1f}s')

    # عرض إحصائيات — Show statistics
    counts = DB.count_by_status(result.get('run_id'))
    print(f'\n📋 Word Status:')
    for status, count in counts.items():
        print(f'   {status}: {count}')


In [ ]:
# 🖥️ واجهة Gradio — Gradio UI

def _get_run_ids():
    """جلب قائمة معرفات التشغيل — Get list of run IDs"""
    with DB._conn() as c:
        rows = c.execute(
            'SELECT run_id, created_at FROM processing_runs ORDER BY created_at DESC LIMIT 20'
        ).fetchall()
        return [str(r['run_id']) for r in rows] or ['']


# --- Tab 1: معالجة المستندات — Document Processing ---

def gradio_process(file, start_pg, end_pg, adaptive_thresh):
    if file is None:
        return '❌ يرجى رفع ملف — Please upload a file', ''
    path = file if isinstance(file, str) else file.name
    result = process_document(
        input_path=path,
        start_page=int(start_pg) if start_pg else 1,
        end_page=int(end_pg) if end_pg else None,
        adaptive=bool(adaptive_thresh)
    )
    if 'error' in result:
        return f'❌ {result["error"]}', ''
    summary = (
        f'✅ تمت المعالجة بنجاح!\n'
        f'Pages: {result.get("pages", 0)}\n'
        f'Words: {result.get("words", 0)}\n'
        f'Avg Confidence: {result.get("avg_confidence", 0):.2%}\n'
        f'Duration: {result.get("duration_sec", 0):.1f}s'
    )
    sentences = reconstruct_sentences(run_id=result.get('run_id'))
    text = '\n'.join(f'--- Page {p} ---\n{s}' for p, s in sentences)
    return summary, text


# --- Tab 2: مراجعة الكلمات — Word Review ---

def gradio_load_words(run_id, status_filter, limit):
    run_id = run_id.strip() if run_id else None
    if status_filter == 'all':
        status_filter = None
    data = DB.get_all(run_id=run_id, status=status_filter)
    if not data:
        return 'No words found', ''
    data = data[:int(limit)] if limit else data
    rows = []
    for i, row in enumerate(data):
        st = row.get('status', '?')
        raw = row.get('raw_text', '')
        cor = row.get('corrected_text', '')
        conf = row.get('confidence', 0)
        eng = row.get('ocr_engine', '')
        rows.append(f'{i+1}. [{st}] {raw} -> {cor} (conf: {conf:.2f}, {eng})')
    return '\n'.join(rows), str(len(data))


def gradio_correct_word(word_id, corrected_text, status):
    try:
        wid = int(word_id)
        DB.update_word(wid, corrected_text, status)
        original = ''
        with DB._conn() as c:
            row = c.execute('SELECT raw_text FROM handwriting_data WHERE id=?', (wid,)).fetchone()
            if row:
                original = row['raw_text']
        append_feedback(word_id, original, corrected_text, status)
        return f'✅ Word {wid} updated to: {corrected_text}'
    except Exception as e:
        return f'❌ Error: {e}'


# --- Tab 3: مراجعة الجمل — Sentence Review ---

def gradio_reconstruct(run_id, verified_only):
    run_id = run_id.strip() if run_id else None
    sentences = reconstruct_sentences(
        run_id=run_id, verified_only=bool(verified_only)
    )
    if not sentences:
        return 'No sentences found'
    result = []
    for page_num, text in sentences:
        result.append(f'--- Page {page_num} ---\n{text}')
    return '\n\n'.join(result)


# --- Tab 4: الإحصائيات — Statistics ---

def gradio_show_stats(run_id):
    run_id = run_id.strip() if run_id else None

    # عدد الكلمات حسب الحالة — Count by status
    counts = DB.count_by_status(run_id=run_id)
    total = sum(counts.values())
    stats = f'📊 Statistics (Total: {total} words)\n\n'
    for status, count in sorted(counts.items()):
        pct = (count / total * 100) if total > 0 else 0
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        stats += f'{status:15s} | {bar} {count} ({pct:.1f}%)\n'

    # مقاييس WER/CER — Metrics
    metrics = compute_metrics(run_id=run_id)
    if 'wer_mean' in metrics:
        stats += f'\n📈 Metrics:\n'
        stats += f'   WER: {metrics["wer_mean"]:.4f} (median: {metrics["wer_median"]:.4f})\n'
        stats += f'   CER: {metrics["cer_mean"]:.4f} (median: {metrics["cer_median"]:.4f})\n'
        stats += f'   Samples: {metrics["num_samples"]}\n'

    return stats


def gradio_plot_confidence(run_id):
    run_id = run_id.strip() if run_id else None
    path = plot_confidence_distribution(run_id=run_id)
    return path if path else None


def gradio_plot_metrics():
    path = plot_metrics_history()
    return path if path else None


# --- Tab 5: التصدير — Export ---

def gradio_export(run_id, fmt):
    run_id = run_id.strip() if run_id else None
    if not run_id:
        return '❌ Select a run ID first'
    output_dir = auto_export(run_id)
    if not output_dir:
        return '❌ Export failed — no data'

    # العثور على آخر ملف — Find latest file
    out_dir = Path(output_dir)
    if fmt == 'csv':
        files = sorted(out_dir.glob('*.csv'), key=lambda f: f.stat().st_mtime, reverse=True)
    elif fmt == 'xlsx':
        files = sorted(out_dir.glob('*.xlsx'), key=lambda f: f.stat().st_mtime, reverse=True)
    elif fmt == 'txt':
        files = sorted(out_dir.glob('*.txt'), key=lambda f: f.stat().st_mtime, reverse=True)
    elif fmt == 'jsonl':
        files = sorted(out_dir.glob('*.jsonl'), key=lambda f: f.stat().st_mtime, reverse=True)
    else:
        files = list(out_dir.iterdir())

    if files:
        return f'✅ Exported to: {output_dir}\nLatest: {files[0].name}'
    return f'✅ Exported to: {output_dir}'


def gradio_upload_hf(repo_id, hf_token):
    return upload_to_hf(repo_id=repo_id, hf_token=hf_token)


# --- Tab 6: التدريب — Training ---

def gradio_check_al():
    status = check_active_learning_trigger()
    msg = f'📊 Active Learning Status:\n'
    msg += f'   Total corrections: {status.get("total_corrections", 0)}\n'
    msg += f'   Recent corrections: {status.get("recent_corrections", 0)}\n'
    msg += f'   Threshold: {status.get("threshold", 0)}\n'
    msg += f'   Trigger: {"✅ Yes" if status.get("trigger") else "❌ No"}\n'
    msg += f'   {status.get("message", "")}'
    return msg


def gradio_finetune():
    result = finetune_trocr_lora()
    if 'error' in result:
        return f'❌ {result["error"]}'
    return (
        f'✅ Fine-tuning complete!\n'
        f'   Epochs: {result.get("epochs", 0)}\n'
        f'   Samples: {result.get("samples_trained", 0)}\n'
        f'   Final Loss: {result.get("final_loss", "N/A")}'
    )


def gradio_reprocess():
    result = reprocess_low_confidence()
    return (
        f'🔄 Reprocess Results:\n'
        f'   Checked: {result.get("checked", 0)}\n'
        f'   Updated: {result.get("updated", 0)}\n'
        f'   Improved: {result.get("improved", 0)}'
    )


# --- Tab 7: القاموس — Dictionary ---

def gradio_show_dict():
    d = load_correction_dict()
    if not d:
        return 'No corrections in dictionary yet.'
    lines = [f'{k} → {v}' for k, v in list(d.items())[:100]]
    return f'📖 Correction Dictionary ({len(d)} entries):\n\n' + '\n'.join(lines)


def gradio_add_correction(original, corrected):
    if not original or not corrected:
        return '❌ Both fields required'
    append_feedback('manual', original, corrected, 'verified')
    global correction_dict
    correction_dict = build_correction_dict()
    return f'✅ Added: "{original}" → "{corrected}" (Dict size: {len(correction_dict)})'


# --- بناء الواجهة — Build Interface ---

with gr.Blocks(
    title='Handwritten OCR — استخراج النصوص اليدوية',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown('# 🖊️ استخراج وتصحيح النصوص اليدوية — Handwritten OCR')
    gr.Markdown('**TrOCR + EasyOCR + Tesseract** | Arabic · English · German')

    with gr.Tab('📄 معالجة المستندات | Document Processing'):
        with gr.Row():
            file_input = gr.File(label='رفع ملف — Upload File (PDF/Image)')
            with gr.Column():
                start_pg = gr.Number(label='Start Page', value=1, precision=0)
                end_pg = gr.Number(label='End Page (empty=all)', value=None, precision=0)
                adaptive = gr.Checkbox(label='Adaptive Threshold', value=False)
        with gr.Row():
            btn_process = gr.Button('🚀 Process — معالجة', variant='primary', size='lg')
        with gr.Row():
            process_summary = gr.Textbox(label='Summary — الملخص', lines=5)
            process_text = gr.Textbox(label='Extracted Text — النص المستخرج', lines=10)
        btn_process.click(
            gradio_process,
            inputs=[file_input, start_pg, end_pg, adaptive],
            outputs=[process_summary, process_text]
        )

    with gr.Tab('✏️ مراجعة الكلمات | Word Review'):
        with gr.Row():
            run_sel1 = gr.Dropdown(label='Run ID', choices=_get_run_ids(), allow_custom_value=True)
            status_sel = gr.Dropdown(
                label='Status', choices=['all', 'pending', 'verified', 'auto_corrected'], value='pending'
            )
            limit_sel = gr.Number(label='Limit', value=50, precision=0)
        btn_load = gr.Button('Load Words — تحميل')
        words_display = gr.Textbox(label='Words — الكلمات', lines=15)
        word_count = gr.Textbox(label='Count — العدد')
        with gr.Row():
            word_id = gr.Number(label='Word ID', precision=0)
            corrected = gr.Textbox(label='Corrected Text — النص المصحح')
            word_status = gr.Dropdown(label='Status', choices=['verified', 'rejected', 'pending'])
        btn_correct = gr.Button('✅ Correct — تصحيح')
        correct_msg = gr.Textbox(label='Result — النتيجة')

        btn_load.click(gradio_load_words, [run_sel1, status_sel, limit_sel], [words_display, word_count])
        btn_correct.click(gradio_correct_word, [word_id, corrected, word_status], correct_msg)

    with gr.Tab('📝 مراجعة الجمل | Sentence Review'):
        with gr.Row():
            run_sel2 = gr.Dropdown(label='Run ID', choices=_get_run_ids(), allow_custom_value=True)
            verified_only = gr.Checkbox(label='Verified Only — المراجعة فقط', value=False)
        btn_recon = gr.Button('Reconstruct — إعادة بناء')
        sentence_display = gr.Textbox(label='Sentences — الجمل', lines=20)
        btn_recon.click(gradio_reconstruct, [run_sel2, verified_only], sentence_display)

    with gr.Tab('📊 الإحصائيات | Statistics'):
        with gr.Row():
            run_sel3 = gr.Dropdown(label='Run ID', choices=_get_run_ids(), allow_custom_value=True)
        btn_stats = gr.Button('Show Stats — عرض الإحصائيات')
        stats_display = gr.Textbox(label='Statistics — الإحصائيات', lines=12)
        with gr.Row():
            btn_conf_plot = gr.Button('Confidence Plot')
            btn_metrics_plot = gr.Button('Metrics Plot')
        with gr.Row():
            conf_plot = gr.Image(label='Confidence Distribution')
            metrics_plot = gr.Image(label='WER/CER History')

        btn_stats.click(gradio_show_stats, run_sel3, stats_display)
        btn_conf_plot.click(gradio_plot_confidence, run_sel3, conf_plot)
        btn_metrics_plot.click(gradio_plot_metrics, [], metrics_plot)

    with gr.Tab('📤 التصدير | Export'):
        with gr.Row():
            run_sel4 = gr.Dropdown(label='Run ID', choices=_get_run_ids(), allow_custom_value=True)
            fmt_sel = gr.Dropdown(
                label='Format', choices=['csv', 'xlsx', 'txt', 'jsonl'], value='csv'
            )
        btn_export = gr.Button('Export — تصدير')
        export_msg = gr.Textbox(label='Result — النتيجة')
        gr.Markdown('### Upload to HuggingFace — رفع إلى HuggingFace')
        with gr.Row():
            hf_repo = gr.Textbox(label='Repo ID', value=CFG.hf_dataset_repo or '')
            hf_tok = gr.Textbox(label='HF Token', type='password')
        btn_hf = gr.Button('🤗 Upload')
        hf_msg = gr.Textbox(label='Result')

        btn_export.click(gradio_export, [run_sel4, fmt_sel], export_msg)
        btn_hf.click(gradio_upload_hf, [hf_repo, hf_tok], hf_msg)

    with gr.Tab('🎯 التدريب | Training'):
        btn_al_check = gr.Button('Check Active Learning — التحقق')
        al_msg = gr.Textbox(label='Status — الحالة', lines=6)
        btn_ft = gr.Button('Fine-tune LoRA — ضبط دقيق', variant='primary')
        ft_msg = gr.Textbox(label='Result — النتيجة', lines=4)
        btn_rp = gr.Button('Reprocess Low Confidence — إعادة معالجة')
        rp_msg = gr.Textbox(label='Result — النتيجة', lines=4)

        btn_al_check.click(gradio_check_al, [], al_msg)
        btn_ft.click(gradio_finetune, [], ft_msg)
        btn_rp.click(gradio_reprocess, [], rp_msg)

    with gr.Tab('📖 القاموس | Dictionary'):
        btn_show_dict = gr.Button('Show Dictionary — عرض القاموس')
        dict_display = gr.Textbox(label='Correction Dictionary — القاموس التصحيحي', lines=15)
        gr.Markdown('### Add Correction — إضافة تصحيح')
        with gr.Row():
            dict_orig = gr.Textbox(label='Original — الأصلي')
            dict_corr = gr.Textbox(label='Corrected — المصحح')
        btn_add_corr = gr.Button('Add — إضافة')
        dict_msg = gr.Textbox(label='Result — النتيجة')

        btn_show_dict.click(gradio_show_dict, [], dict_display)
        btn_add_corr.click(gradio_add_correction, [dict_orig, dict_corr], dict_msg)

# --- تشغيل الواجهة — Launch Interface ---
demo.launch(share=CFG.gradio_share, debug=False)
